# 0. Introduction and Problem Definition

## Use case

Before starting, we answered three questions:

- **Who would use this model, and for what?** An architect or a small developer in King County doing a feasibility study. They compare design options for a house (size, bedrooms, bathrooms, floors, construction quality) and want a quick estimate of the market value of each option.
- **Which mistake is more expensive?** Over-estimating the price. 
- **What would the user know at prediction time?** The design of the house (living area, bedrooms, bathrooms, floors, basement, grade, condition, year built and renovated) and the plot (lot area, ZIP code and coordinates). 

In [2]:
%pip install -r ../requirements.txt

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## Setup

In [3]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib   # to save and load the trained model

### Folder Directory

In [4]:
# --- Folders --------------------------------------------------------------
DATA_DIR = Path("..") / "data"      # the clean CSV file from Day 1 is here
FIG_DIR = Path("..") / "figures"    # exported plots go here
MODEL_DIR = Path("..") / "models"   # the trained model is saved here
for folder in [DATA_DIR, FIG_DIR, MODEL_DIR]:
    folder.mkdir(exist_ok=True)

# --- pandas display options -------------------------------------------------
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:,.2f}".format)

# --- plot style -------------------------------------------------------------
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (10, 5)

# Seed for anything random (split, cross-validation, models),
# so we always get the same results.
RANDOM_STATE = 42


def save_fig(name):
    # Save the current figure as a PNG inside FIG_DIR.
    path = FIG_DIR / f"{name}.png"
    plt.savefig(path, dpi=150, bbox_inches="tight")
    print(f"Figure saved to {path}")

## Load the Clean Dataset from Day 1

We start from the clean dataset we produced on Day 1 (`kc_houses_data_clean.csv`). On Day 1 we:

- converted the `date` column from a string to a datetime format,
- dropped the `waterfront` and `view` columns, because their values are heavily concentrated at zero,
- removed the records below the 1st and above the 99th percentile of price, living area and bedroom count (910 records removed).

In [5]:
clean_path = DATA_DIR / "kc_houses_data_clean.csv"

if not clean_path.exists():
    raise FileNotFoundError(
        f"{clean_path} not found. Run the Day 1 notebook first and copy "
        "data/kc_houses_data_clean.csv into this repository's data/ folder."
    )

houses = pd.read_csv(clean_path, parse_dates=["date"])
print(f"Loaded {houses.shape[0]:,} rows and {houses.shape[1]} columns")
houses.head()

Loaded 20,703 rows and 19 columns


,id,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,condition,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15
0,7129300520,2014-10-13,"221,900.00",3,1.00,1180,5650,1.00,3,7,1180,0,1955,0,98178,47.51,-122.26,1340,5650
1,6414100192,2014-12-09,"538,000.00",3,2.25,2570,7242,2.00,3,7,2170,400,1951,1991,98125,47.72,-122.32,1690,7639
2,5631500400,2015-02-25,"180,000.00",2,1.00,770,10000,1.00,3,6,770,0,1933,0,98028,47.74,-122.23,2720,8062
3,2487200875,2014-12-09,"604,000.00",4,3.00,1960,5000,1.00,5,7,1050,910,1965,0,98136,47.52,-122.39,1360,5000
4,1954400510,2015-02-18,"510,000.00",3,2.00,1680,8080,1.00,3,8,1680,0,1987,0,98074,47.62,-122.05,1800,7503


In [6]:
houses.info()

<class 'pandas.DataFrame'>
RangeIndex: 20703 entries, 0 to 20702
Data columns (total 19 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   id             20703 non-null  int64         
 1   date           20703 non-null  datetime64[us]
 2   price          20703 non-null  float64       
 3   bedrooms       20703 non-null  int64         
 4   bathrooms      20703 non-null  float64       
 5   sqft_living    20703 non-null  int64         
 6   sqft_lot       20703 non-null  int64         
 7   floors         20703 non-null  float64       
 8   condition      20703 non-null  int64         
 9   grade          20703 non-null  int64         
 10  sqft_above     20703 non-null  int64         
 11  sqft_basement  20703 non-null  int64         
 12  yr_built       20703 non-null  int64         
 13  yr_renovated   20703 non-null  int64         
 14  zipcode        20703 non-null  int64         
 15  lat            20703 non-null 

## Problem Type, Target and Main Metric

- **Problem type:** regression, because we predict a number.
- **Target:** `price`, the sale price of the house in USD.
- **Main metric:** MAE (mean absolute error). It shows how many dollars the prediction is wrong on average, which is easy to understand for our user.

We also look at RMSE, R² and MAPE (the error in %). Since over-estimating the price is the more expensive mistake, we will also check how often the model predicts too high.

# 1. Features, Leakage and Split

In this section we decide which columns the model can use as inputs (X) and which column it has to predict (y). X contains the inputs, the information about the house that the model uses. y is the target, the sale price that the model tries to predict. Then we split the data into a training set and a test set.

## 1.1 Removed Columns

We only keep the columns that the user knows before the house is sold. We removed these columns from the inputs:

- `price`: This is the target.
- `id`: Only an identifier and says nothing about the house.
- `date`: The sale date is not part of the design, and the data covers only one year.
- `sqft_above`: Same as `sqft_living` minus `sqft_basement`, so it is repeated information.
- `sqft_living15`, `sqft_lot15`: Measured on the neighbouring houses in 2015. A designer would not know them, and using them could cause leakage.

The columns `waterfront` and `view` were already dropped on Day 1.

In [ ]:
# Check sqft_above can be calculated from two other columns
is_redundant = (houses["sqft_above"] == houses["sqft_living"] - houses["sqft_basement"]).all()
print(f"sqft_above = sqft_living - sqft_basement in every row: {is_redundant}")

sqft_above = sqft_living - sqft_basement in every row: True


In [8]:
TARGET = "price"
columns_to_drop = ["price", "id", "date", "sqft_above", "sqft_living15", "sqft_lot15"]

X = houses.drop(columns=columns_to_drop)   # inputs
y = houses[TARGET]                         # target
groups = houses["id"]                      # only used to build the split, not as an input

print(f"X: {X.shape[0]:,} rows and {X.shape[1]} columns")
print(f"Input columns: {X.columns.tolist()}")

X: 20,703 rows and 13 columns
Input columns: ['bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'floors', 'condition', 'grade', 'sqft_basement', 'yr_built', 'yr_renovated', 'zipcode', 'lat', 'long']


## 1.2 Predicting the Log of the Price

On Day 1 we saw that prices are right-skewed: the median is about 450,000 USD and the mean is about 519,000 USD. (A few very expensive houses pull the mean up.)

Errors on house prices also grow with the price: an error of 50,000 USD is large for a cheap house but small for an expensive one. If we predict **log(price)** instead of the price, the target becomes almost symmetric and the errors become more similar in size. 


In [10]:
print(f"Skewness of price:      {y.skew():.2f}")
print(f"Skewness of log(price): {np.log(y).skew():.2f}")

Skewness of price:      1.72
Skewness of log(price): 0.28


## 1.3 Train / Test Split

Some houses were sold twice during the year, so the same `id` appears in two rows. If one sale is in the training set and the other in the test set, the model has already seen the house.

To avoid this, we use `GroupShuffleSplit` with `id` as the group, so both sales of the same house stay in the same set. We use 80% of the data for training and 20% for testing, with a fixed `random_state`.


In [11]:
from sklearn.model_selection import GroupShuffleSplit

n_repeated = houses["id"].duplicated(keep=False).sum()
print(f"Sales of houses that were sold more than once: {n_repeated}")

splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
train_idx, test_idx = next(splitter.split(X, y, groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
groups_train = groups.iloc[train_idx]   # needed for cross-validation in section 3

# Check that no house is in both sets
houses_in_both = set(groups.iloc[train_idx]) & set(groups.iloc[test_idx])
print(f"Train set: {len(X_train):,} rows")
print(f"Test set:  {len(X_test):,} rows")
print(f"Houses in both sets: {len(houses_in_both)}")

Sales of houses that were sold more than once: 296
Train set: 16,556 rows
Test set:  4,147 rows
Houses in both sets: 0


# 2. Preprocessing Pipeline

## 2.1 Type of Each Input Column

- `zipcode`: a category stored as a number (70 ZIP codes, see Day 1). ZIP code 98103 is not "bigger" than 98023, so we use one-hot encoding.
- `grade` (1–13) and `condition` (1–5): ordered categories. A higher number means better quality, so we keep them as numbers.
- All other columns (areas, counts, years and coordinates) are numeric.

## 2.2 Building the Pipeline

We put all preprocessing steps inside a `Pipeline` and a `ColumnTransformer`:

- Imputation: in case a user leaves a field empty.
- Scaling: only for the linear model and KNN. Tree models do not need it.
- One-hot encoding for `zipcode`.

Because everything is inside the pipeline, these steps are fitted only on the training data. (Nothing leaks into the test set.)

In [12]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

numeric_columns = ["bedrooms", "bathrooms", "sqft_living", "sqft_lot", "floors",
                   "condition", "grade", "sqft_basement", "yr_built", "yr_renovated",
                   "lat", "long"]
nominal_columns = ["zipcode"]


def make_preprocessor(scale):
    # Numeric columns: fill missing values with the median, scale if needed
    numeric_steps = [("impute", SimpleImputer(strategy="median"))]
    if scale:
        numeric_steps.append(("scale", StandardScaler()))
    numeric_pipeline = Pipeline(numeric_steps)

    # ZIP code: fill missing values with the most frequent one, then one-hot encode
    nominal_pipeline = Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ])

    return ColumnTransformer([
        ("numeric", numeric_pipeline, numeric_columns),
        ("nominal", nominal_pipeline, nominal_columns),
    ])

In [13]:
# Check the result on the training set
preview = make_preprocessor(scale=True).set_output(transform="pandas")
X_train_preview = preview.fit_transform(X_train)

print(f"Before preprocessing: {X_train.shape[1]} columns")
print(f"After preprocessing:  {X_train_preview.shape[1]} columns")
X_train_preview.iloc[:3, :14]

Before preprocessing: 13 columns
After preprocessing:  82 columns


,numeric__bedrooms,numeric__bathrooms,numeric__sqft_living,numeric__sqft_lot,numeric__floors,numeric__condition,numeric__grade,numeric__sqft_basement,numeric__yr_built,numeric__yr_renovated,numeric__lat,numeric__long,nominal__zipcode_98001,nominal__zipcode_98002
0,-0.46,-1.55,-1.09,-0.23,-0.92,-0.64,-0.60,-0.67,-0.57,-0.21,-0.35,-0.32,0.00,0.00
1,-0.46,0.20,0.64,-0.19,0.93,-0.64,-0.60,0.28,-0.71,4.79,1.15,-0.76,0.00,0.00
2,-1.66,-1.55,-1.60,-0.12,-0.92,-0.64,-1.52,-0.67,-1.32,-0.21,1.28,-0.15,0.00,0.00
